In [0]:
df = spark.read.format("parquet")\
    .load("abfss://bronze@monarchazuredatalake.dfs.core.windows.net/orders")

df.display()


#### droping the collumn, changing the name 

In [0]:
df.schema
#for reanming
df = df.withColumnRenamed("_rescued_data","rescued_data")
# now we can drop the data - after the bronze layer - why?
df = df.drop("rescued_data")
df.display()

In [0]:
#changing the data formate - DATE format

from pyspark.sql.functions import *
## withColumn will check if the column is present then it udates it or if it is not then it creates a new column
df = df.withColumn("order_date",to_timestamp(col('order_date')))


In [0]:
# create a new column, 
df = df.withColumn("year", year(col("order_date")))
df.display()

###### Windows function

In [0]:
from pyspark.sql.window import Window 

df1 = df.withColumn("flag", dense_rank().over(Window.partitionBy("year").orderBy(desc("total_amount"))))

###### OOPS CLASS

###### Data Writing

writing into the data lake silver layer

In [0]:
df1.write.format("delta").save("abfss://silver@monarchazuredatalake.dfs.core.windows.net/orders")